# IXI Data Train/Val Split (90/10)

**Purpose**: Split your preprocessed IXI data into training (90%) and validation (10%) sets.

**Run this ONCE before training your models.**

---

## What This Does

- Takes all `.npy` files from your `ixi_resized` folder
- Randomly splits into 90% train / 10% validation
- Creates organized folder structure: `processed_ixi/train/` and `processed_ixi/val/`
- Verifies the split worked correctly

**Expected Result:**
- If you have ~16,771 slices → ~15,094 train, ~1,677 validation

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Configure Paths

**⚠️ UPDATE THE SOURCE_FOLDER PATH TO MATCH YOUR DRIVE STRUCTURE!**

In [ ]:
import os
import shutil
import random
import numpy as np
from pathlib import Path

# ============================================
# CONFIGURATION - Update this!
# ============================================

# Path to your current resized IXI data (all files in one folder)
# Examples:
#   '/content/drive/MyDrive/IXI_Project/ixi_resized'
#   '/content/drive/MyDrive/symAD-ECNN/ixi_resized'
SOURCE_FOLDER = '/content/drive/MyDrive/YOUR_FOLDER_NAME/ixi_resized'  # ← UPDATE THIS!

# Destination folder for split data
DEST_BASE = '/content/drive/MyDrive/symAD-ECNN/data/processed_ixi'

# Split ratio (90% train, 10% validation)
TRAIN_RATIO = 0.9

print("✓ Configuration set")
print(f"  Source: {SOURCE_FOLDER}")
print(f"  Destination: {DEST_BASE}")
print(f"  Split: {int(TRAIN_RATIO*100)}% train, {int((1-TRAIN_RATIO)*100)}% validation")

## Step 3: Create Folder Structure

In [ ]:
# Create train and val directories
train_dir = os.path.join(DEST_BASE, 'train')
val_dir = os.path.join(DEST_BASE, 'val')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

print("✓ Created folder structure:")
print(f"  Train: {train_dir}")
print(f"  Val: {val_dir}")

## Step 4: Check Source Data

In [ ]:
# Check if source folder exists
if not os.path.exists(SOURCE_FOLDER):
    print(f"❌ ERROR: Source folder not found!")
    print(f"   Path: {SOURCE_FOLDER}")
    print(f"\nPlease update SOURCE_FOLDER in Step 2 above.")
    print(f"\nYour Drive folders:")
    for folder in os.listdir('/content/drive/MyDrive'):
        print(f"  - {folder}")
else:
    # Get all .npy files
    all_files = [f for f in os.listdir(SOURCE_FOLDER) if f.endswith('.npy')]
    total_files = len(all_files)
    
    if total_files == 0:
        print(f"❌ ERROR: No .npy files found in {SOURCE_FOLDER}")
        print(f"\nFiles in folder:")
        for item in os.listdir(SOURCE_FOLDER)[:10]:
            print(f"  - {item}")
    else:
        print(f"✓ Found {total_files:,} .npy files in source folder")
        
        # Check sample file
        sample_file = os.path.join(SOURCE_FOLDER, all_files[0])
        sample_data = np.load(sample_file)
        print(f"✓ Sample shape: {sample_data.shape}")
        print(f"✓ Sample dtype: {sample_data.dtype}")

## Step 5: Split Data

This will randomly split your data into train/val sets.

In [ ]:
# Shuffle files for random split (with fixed seed for reproducibility)
random.seed(42)
random.shuffle(all_files)

# Calculate split point
split_idx = int(total_files * TRAIN_RATIO)

train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

print(f"Data split calculated:")
print(f"  Train: {len(train_files):,} files ({len(train_files)/total_files*100:.1f}%)")
print(f"  Val: {len(val_files):,} files ({len(val_files)/total_files*100:.1f}%)")
print(f"  Total: {len(train_files) + len(val_files):,} files")

## Step 6: Choose Copy or Move

**Copy (Recommended)**: Keeps original files (safer, uses more space)  
**Move**: Deletes original files (saves space, but can't undo)

In [ ]:
# Choose operation
USE_COPY = True  # Set to False to MOVE files instead

if USE_COPY:
    copy_func = shutil.copy2
    action = "Copying"
    print("✓ Will COPY files (original will be kept)")
else:
    copy_func = shutil.move
    action = "Moving"
    print("⚠️  Will MOVE files (original will be deleted!)")

## Step 7: Copy/Move Training Files

⏱️ This may take 5-10 minutes depending on file count.

In [ ]:
print(f"{action} {len(train_files):,} training files...")
print("(Progress updates every 1000 files)\n")

for i, filename in enumerate(train_files):
    src = os.path.join(SOURCE_FOLDER, filename)
    dst = os.path.join(train_dir, filename)
    copy_func(src, dst)
    
    if (i + 1) % 1000 == 0:
        progress = (i + 1) / len(train_files) * 100
        print(f"  Progress: {i + 1:,}/{len(train_files):,} ({progress:.1f}%)")

print(f"\n✅ Completed! {len(train_files):,} files in train folder")

## Step 8: Copy/Move Validation Files

In [ ]:
print(f"{action} {len(val_files):,} validation files...")
print("(Progress updates every 100 files)\n")

for i, filename in enumerate(val_files):
    src = os.path.join(SOURCE_FOLDER, filename)
    dst = os.path.join(val_dir, filename)
    copy_func(src, dst)
    
    if (i + 1) % 100 == 0:
        progress = (i + 1) / len(val_files) * 100
        print(f"  Progress: {i + 1:,}/{len(val_files):,} ({progress:.1f}%)")

print(f"\n✅ Completed! {len(val_files):,} files in val folder")

## Step 9: Verify Split

In [ ]:
print("="*50)
print("VERIFICATION")
print("="*50 + "\n")

# Count files in each folder
train_count = len([f for f in os.listdir(train_dir) if f.endswith('.npy')])
val_count = len([f for f in os.listdir(val_dir) if f.endswith('.npy')])
total_split = train_count + val_count

print(f"Files in train folder: {train_count:,}")
print(f"Files in val folder: {val_count:,}")
print(f"Total after split: {total_split:,}")
print(f"Original total: {total_files:,}")

if not USE_COPY:
    remaining = len([f for f in os.listdir(SOURCE_FOLDER) if f.endswith('.npy')])
    print(f"Files remaining in source: {remaining:,}")

# Check if numbers match
if total_split == total_files:
    print("\n✅ SUCCESS! All files accounted for.")
else:
    print(f"\n⚠️ WARNING: File count mismatch!")
    print(f"  Expected: {total_files:,}")
    print(f"  Got: {total_split:,}")
    print(f"  Difference: {abs(total_split - total_files):,}")

## Step 10: Sample Check

In [ ]:
print("="*50)
print("SAMPLE CHECK")
print("="*50 + "\n")

try:
    # Load sample from train
    train_sample_file = os.path.join(train_dir, os.listdir(train_dir)[0])
    train_sample = np.load(train_sample_file)
    print(f"✓ Train sample loaded")
    print(f"  Shape: {train_sample.shape}")
    print(f"  Dtype: {train_sample.dtype}")
    print(f"  Range: [{train_sample.min():.3f}, {train_sample.max():.3f}]")
    
    # Load sample from val
    val_sample_file = os.path.join(val_dir, os.listdir(val_dir)[0])
    val_sample = np.load(val_sample_file)
    print(f"\n✓ Val sample loaded")
    print(f"  Shape: {val_sample.shape}")
    print(f"  Dtype: {val_sample.dtype}")
    print(f"  Range: [{val_sample.min():.3f}, {val_sample.max():.3f}]")
    
    print("\n✅ All samples loaded successfully!")
    
except Exception as e:
    print(f"❌ ERROR loading samples: {e}")

## Final Summary

In [ ]:
print("\n" + "="*50)
print("✅ DATA SPLIT COMPLETE!")
print("="*50 + "\n")

print(f"Training set: {train_count:,} samples ({train_count/total_files*100:.1f}%)")
print(f"Validation set: {val_count:,} samples ({val_count/total_files*100:.1f}%)")
print(f"\nData location:")
print(f"  Train: {train_dir}")
print(f"  Val: {val_dir}")

print("\n" + "="*50)
print("NEXT STEPS")
print("="*50 + "\n")

print("1. Update your training notebooks with these paths:")
print(f"   IXI_TRAIN_PATH = '{train_dir}'")
print(f"   IXI_VAL_PATH = '{val_dir}'")
print("\n2. Upload BraTS data to Drive (if not already done)")
print("\n3. Start training your models!")

if USE_COPY:
    print("\n4. (Optional) Delete source folder to save space:")
    print(f"   {SOURCE_FOLDER}")

print("\n✅ Ready for training!")